### Datos faltantes: análisis inicial

Identificación de columnas con valores nulos y su posible relación con otras variables.

In [ ]:
import pandas as pd 
import numpy as np
from IPython.display import display
import sys, os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import mannwhitneyu
import missingno as msno
%matplotlib inline
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
#from functions.transformers import *

In [ ]:
df_train_raw = pd.read_csv('../data/raw/train.csv').drop('Id', axis=1)
df_train = df_train_raw.copy()
print(f'start dimensions: {df_train.shape}')
df_train.head()

In [ ]:
missing_data = df_train.isnull().sum()
missing_data = pd.DataFrame(missing_data[missing_data > 0], columns=['missing counts'])
missing_data['percentage(%)'] = np.round(missing_data['missing counts'] / df_train.shape[0] * 100, 2)
missing_data = missing_data.sort_values(by='percentage(%)', ascending=False)
missing_data

In [ ]:
#PoolQC: Calidad de la piscina.
print(f"-- PoolQC ---")
#Visualización de los datos faltantes, cantidad y porcentaje
print(df_train['PoolQC'].unique())
missing_poolqc = pd.DataFrame(df_train['PoolQC'].value_counts(dropna=False))
missing_poolqc['percentage(%)'] = np.round(missing_poolqc['count'] / df_train.shape[0] * 100, 2)
display(missing_poolqc)
#Visualización de los datos faltantes, area de piscina frente a calidad de piscina
res = df_train[df_train['PoolArea'] > 0]
display(res[['PoolArea', 'PoolQC']])
#Imputación de los datos faltantes
df_train.loc[:, 'PoolQC'] = df_train['PoolQC'].fillna('NA')
#Visualización de la relación entre PoolQC y PoolArea graficamente
plt.figure(figsize=(8,4))
sns.violinplot(x='PoolQC', y='PoolArea', data=df_train)
plt.title('Pool Quality vs Pool Area')
plt.show()
#PoolQC: Calidad de la piscina. depende de PoolArea. Si PoolArea es 0, PoolQC es NA.
#MNAR Las features estan relacionadas. Se puede imputar. No son aleatorias.

In [ ]:
#MiscFeature: Característica miscelánea no cubierta en otras categorías.
print(f"-- MiscFeature ---")
#MiscVal: Valor monetario de la característica miscelánea.
#Visualización de los datos faltantes, cantidad y porcentaje
missing_miscFeature = pd.DataFrame(df_train['MiscFeature'].value_counts(dropna=False))
missing_miscFeature['percentage(%)'] = np.round(missing_miscFeature['count'] / df_train.shape[0] * 100, 2)
display(missing_miscFeature)
#visualizar las filas con datos faltantes en MiscFeature y su relación con MiscVal
match_miscval = df_train[df_train['MiscVal'] > 0]
display(match_miscval[['MiscFeature', 'MiscVal']])
#Imputación de los datos faltantes
df_train.loc[:, 'MiscFeature'] = df_train['MiscFeature'].fillna('NA')
#Visualizacion de la relación entre MiscFeature y MiscVal graficamente
plt.figure(figsize=(8,4))
sns.violinplot(x='MiscFeature', y='MiscVal', data=df_train)
plt.title('Misc Feature vs Misc Val')
plt.show()
#MiscFeature: Característica miscelánea no cubierta en otras categorías. Depende de MiscVal. Si MiscVal es 0, MiscFeature es NA.
#MNAR Las features estan relacionadas. Se puede imputar. No son aleatorias.

In [ ]:
#Alley: Tipo de acceso por callejón a la propiedad.
print(f"-- Alley ---")
#Visualización de los datos faltantes, cantidad y porcentaje
missing_alley = pd.DataFrame(df_train['Alley'].value_counts(dropna=False))
missing_alley['percentage(%)'] = np.round(missing_alley['count'] / df_train.shape[0] * 100, 2)
display(missing_alley)
#visualizar las filas con datos faltantes en Alley y su relación con LotFrontage
df_train.loc[:, 'Alley'] = df_train['Alley'].fillna('NA')
df_train.groupby(['Alley'])['LotArea'].agg(['count', 'mean']).reset_index()
#Imputación de los datos faltantes con valor NA, que indica que no tiene callejon.
#Se analizo una posible MCAR, pero no se encontró evidencia suficiente.
#Se determino en la comparacion que tiene un manejo diferente los grupos donde Alley es NA y donde no lo es.
#Se define como MAR, ya que las casas sin callejon tienen a ser mas caras

In [ ]:
""" def test_mcar(df, variable_con_nulos, target_var='SalePrice'):
    # 1. Crear una copia temporal con la bandera de nulos
    temp_df = df.copy()
    temp_df['is_null'] = temp_df[variable_con_nulos].isnull()
    
    # 2. Separar los dos grupos
    grupo_nulos = temp_df[temp_df['is_null'] == True][target_var]
    grupo_datos = temp_df[temp_df['is_null'] == False][target_var]
    
    # 3. Visualización
    plt.figure(figsize=(12, 5))
    
    # Gráfico de densidad (KDE)
    plt.subplot(1, 2, 1)
    sns.kdeplot(grupo_datos, label='Con Datos', shade=True)
    sns.kdeplot(grupo_nulos, label='Nulos (NaN)', shade=True)
    plt.title(f'Distribución de {target_var}\nsegún nulidad de {variable_con_nulos}')
    plt.legend()
    
    # Boxplot para ver medianas y outliers
    plt.subplot(1, 2, 2)
    sns.boxplot(data=temp_df, x='is_null', y=target_var)
    plt.title(f'Comparación de Medianas')
    
    plt.tight_layout()
    plt.show()
    
    # 4. Prueba Estadística (Mann-Whitney U)
    # Es mejor que la t-test porque no asume que los precios son normales
    stat, p_value = mannwhitneyu(grupo_datos, grupo_nulos)
    
    print(f"--- Diagnóstico para {variable_con_nulos} ---")
    print(f"P-Valor de la prueba Mann-Whitney: {p_value:.4f}")
    
    if p_value > 0.05:
        print("Resultado: No hay diferencia significativa. Posible MCAR (Aleatorio).")
    else:
        print("Resultado: Diferencia significativa detectada. Es MAR o MNAR (No aleatorio).")

# Ejemplo de uso:
test_mcar(df_train, 'Alley') """

In [ ]:
missing_data = df_train.isnull().sum()
missing_data = pd.DataFrame(missing_data[missing_data > 0], columns=['missing counts'])
missing_data['percentage(%)'] = np.round(missing_data['missing counts'] / df_train.shape[0] * 100, 2)
missing_data = missing_data.sort_values(by='percentage(%)', ascending=False)
missing_data